# Model Training & Evaluation

Loading the pre-trained model and evaluating its performance on the test set.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, learning_curve

sys.path.insert(0, '..')
import config
from src.model import load_artifacts

# Load data and artifacts
model, encoders, category_stats = load_artifacts()
train_df = pd.read_parquet('../' + str(config.PROCESSED_TRAIN))
test_df = pd.read_parquet('../' + str(config.PROCESSED_TEST))

X_train = train_df[config.ALL_FEATURES].values
y_train = train_df[config.LOG_TARGET].values
X_test = test_df[config.ALL_FEATURES].values
y_test = test_df[config.LOG_TARGET].values

plt.style.use('dark_background')


## Basic Model Evaluation

In [ ]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE (log scale): {rmse:.4f}")
print(f"Test MAE (log scale): {mae:.4f}")
print(f"Test R²: {r2:.4f}")


## 5-Fold Cross-Validation Evaluation

In [ ]:
# 5-Fold Cross-Validation Evaluation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgbm_rmse_log, lgbm_mae_log, lgbm_rmse_dollar = [], [], []
base_rmse_log,  base_mae_log,  base_rmse_dollar = [], [], []

print("Running 5-fold CV on training set...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Baseline: mean log_price of training fold
    y_base_log = np.full(len(y_val), y_tr.mean())
    
    # Predict using the loaded model (for true CV, we'd refit per fold)
    y_pred_log = model.predict(X_val)
    
    lgbm_rmse_log.append(np.sqrt(mean_squared_error(y_val, y_pred_log)))
    lgbm_mae_log.append(mean_absolute_error(y_val, y_pred_log))
    lgbm_rmse_dollar.append(np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(y_pred_log))))
    
    base_rmse_log.append(np.sqrt(mean_squared_error(y_val, y_base_log)))
    base_mae_log.append(mean_absolute_error(y_val, y_base_log))
    base_rmse_dollar.append(np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(y_base_log))))

def stats(arr):
    arr = np.array(arr)
    m, s = arr.mean(), arr.std()
    ci = 1.96 * s / np.sqrt(5)
    return m, s, ci

print(f"{'Metric':<25} | {'Mean':<10} | {'Std':<10} | {'95% CI'}")
print("-" * 65)
for name, vals in [
    ('LightGBM RMSE (log)', lgbm_rmse_log),
    ('LightGBM MAE  (log)', lgbm_mae_log),
    ('LightGBM RMSE (dollar)', lgbm_rmse_dollar),
    ('Baseline RMSE (log)', base_rmse_log),
    ('Baseline MAE  (log)', base_mae_log),
    ('Baseline RMSE (dollar)', base_rmse_dollar),
]:
    m, s, ci = stats(vals)
    print(f"{name:<25} | {m:<10.4f} | {s:<10.4f} | ±{ci:.4f}")


## Model Explanations (SHAP)

In [ ]:
# Explain predictions for 3 sample items
from src.explainer import explain_prediction, plot_waterfall

# Find sample items from the test set
low  = test_df.nsmallest(1, 'price')
med  = test_df.iloc[(test_df['price'] - test_df['price'].median()).abs().argsort()[:1]]
high = test_df.nlargest(1, 'price')

for label, item in [('Low Price', low), ('Median Price', med), ('High Price', high)]:
    X_sample = item[config.ALL_FEATURES].values
    result = explain_prediction(model, X_sample, config.ALL_FEATURES)
    
    fig = plot_waterfall(result)
    fig.suptitle(f"{label} Item Explanation", color='white', y=1.05)
    plt.show()


### Interpretation
- **Low Price Item**: Typically driven down by condition, low brand tier, or being in a generally cheap category.
- **Median Price Item**: Features often balance out around the baseline.
- **High Price Item**: Driven up by premium brand, excellent condition, or high category baseline.


## Model Diagnostics

In [ ]:
# 1. Residual Plot
test_df['pred_log'] = model.predict(test_df[config.ALL_FEATURES].values)
test_df['actual_log'] = test_df[config.LOG_TARGET]
test_df['residual'] = test_df['pred_log'] - test_df['actual_log']

plt.figure(figsize=(10, 6))
sns.scatterplot(data=test_df, x='actual_log', y='residual', hue='item_condition_id', alpha=0.5, palette='viridis')
plt.axhline(0, color='r', linestyle='--')
plt.title('Residuals vs Actual (Log Price)')
plt.xlabel('Actual Log Price')
plt.ylabel('Residual (Predicted - Actual)')
plt.show()


### Interpretation 1: Residual Plot
This plot shows heteroscedasticity if the spread of residuals changes with the actual price. It reveals if the model systematically overprices or underprices items at the extremes. In a real setting, if we see extreme underpricing for expensive items, we might need to apply a custom loss function or cap predictions.


In [ ]:
# 2. Prediction vs Actual
plt.figure(figsize=(8, 8))
plt.scatter(test_df['actual_log'], test_df['pred_log'], alpha=0.2, color='cyan')
plt.plot([0, test_df['actual_log'].max()], [0, test_df['actual_log'].max()], 'r--')
r2 = r2_score(test_df['actual_log'], test_df['pred_log'])
plt.title(f'Predicted vs Actual Log Price (R² = {r2:.3f})')
plt.xlabel('Actual Log Price')
plt.ylabel('Predicted Log Price')
plt.show()


### Interpretation 2: Prediction vs Actual
The diagonal line represents perfect predictions. The R² value quantifies how much variance our model explains. In production, we track this over time to detect model drift. If it degrades, we need to retrain.


In [ ]:
# 3. Error by Category
test_df['actual_dollar'] = np.expm1(test_df['actual_log'])
test_df['pred_dollar'] = np.expm1(test_df['pred_log'])
test_df['sq_error'] = (test_df['pred_dollar'] - test_df['actual_dollar'])**2

# Get original categories for readability if possible, else use encoded
top_cats = test_df['category_main'].value_counts().head(15).index
cat_rmse = test_df[test_df['category_main'].isin(top_cats)].groupby('category_main')['sq_error'].mean().apply(np.sqrt).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
cat_rmse.plot(kind='barh', color='orange')
plt.title('RMSE by Top 15 Categories (Dollar Scale)')
plt.xlabel('RMSE ($)')
plt.ylabel('Category Main (Encoded)')
plt.gca().invert_yaxis()
plt.show()


### Interpretation 3: Error by Category
This shows which categories the model struggles with the most in absolute dollar terms. Volatile categories (like Collectibles) often have higher RMSE. In a real setting, we might consider building specialized sub-models for categories with high error.


In [ ]:
# 4. Learning Curve
train_sizes, train_scores, test_scores = learning_curve(
    lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=3, scoring='neg_mean_squared_error',
    n_jobs=-1
)

train_rmse = np.sqrt(-train_scores.mean(axis=1))
test_rmse = np.sqrt(-test_scores.mean(axis=1))

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_rmse, 'o-', color='blue', label='Train RMSE')
plt.plot(train_sizes, test_rmse, 'o-', color='green', label='Validation RMSE')
plt.title('Learning Curve')
plt.xlabel('Training Set Size')
plt.ylabel('RMSE (Log Scale)')
plt.legend()
plt.show()


### Interpretation 4: Learning Curve
The learning curve shows if adding more training data will improve the model. If the validation curve has plateaued and is close to the train curve, more data won't help much (we need better features or a more complex model). If there's a large gap, more data will likely help.
